# Tabular

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

# Concatenate dataframes
data = pd.concat([sep_data, nsep_data])

# Extract features and target
X = data[['Sunspots', 'AR', 'Flare Class']]
y = data['label']

# Extract Flare Class types without intensities
X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')

# Create dummy variables for Flare Class
flare_dummies = pd.get_dummies(X['Flare Class'], prefix='Flare')
X = pd.concat([X[['Sunspots', 'AR']], flare_dummies], axis=1)

print(X)

# Initialize SVM classifier
svm_classifier = SVC(kernel='linear')

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# List to store accuracy values
accuracy_values_SVM = []

# Perform 5-fold cross-validation
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train the classifier
    svm_classifier.fit(X_train, y_train)

    # Make predictions
    y_pred = svm_classifier.predict(X_test)

    # Evaluate the accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Append accuracy value to the list
    accuracy_values_SVM.append(accuracy)

# Calculate the mean accuracy
mean_accuracy = sum(accuracy_values_SVM) / len(accuracy_values_SVM)
print("Mean Accuracy:", mean_accuracy)

# Print all accuracy values
print("All Accuracy Values:", accuracy_values_SVM)


    Sunspots  AR  Flare_C  Flare_M  Flare_X
0         28  15        0        0        1
1         23   7        0        0        1
2          8   6        0        1        0
3         26   9        0        0        1
4         19  20        1        0        0
..       ...  ..      ...      ...      ...
32        30  16        0        1        0
33        22  15        0        1        0
34        44  25        0        1        0
35        21  10        0        1        0
36        23  22        0        1        0

[74 rows x 5 columns]
Mean Accuracy: 0.7038095238095238
All Accuracy Values: [0.6666666666666666, 0.7333333333333333, 0.8, 0.5333333333333333, 0.7857142857142857]


C:\Users\shiva\AppData\Local\Temp\ipykernel_9052\139778236.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')


In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

# Concatenate dataframes
data = pd.concat([sep_data, nsep_data])

# Extract features and target
X = data[['Sunspots', 'AR', 'Flare Class']]
y = data['label']

# Extract flare class letters
X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')

# One-hot encode flare classes
flare_dummies = pd.get_dummies(X['Flare Class'], prefix='Flare')
X = pd.concat([X[['Sunspots', 'AR']], flare_dummies], axis=1)

# Initialize classifier
# rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier = RandomForestClassifier(n_estimators=50, max_depth=6, min_samples_split=4,
                                       min_samples_leaf=3, max_features="sqrt", class_weight="balanced",
                                       random_state=42)

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Metrics lists
accuracy_values_RF = []
f1_scores_RF = []
tss_scores_RF = []
hss_scores_RF = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_classifier.fit(X_train, y_train)
    y_pred = rf_classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    tss = (tp / (tp + fn)) - (fp / (fp + tn))
    hss = (2 * ((tp * tn) - (fn * fp))) / (((tp + fn) * (fn + tn)) + ((fp + tn) * (tp + fp)))

    accuracy_values_RF.append(accuracy)
    f1_scores_RF.append(f1)
    tss_scores_RF.append(tss)
    hss_scores_RF.append(hss)

# Results
print("\nRandom Forest Results:")
print("Mean Accuracy:", sum(accuracy_values_RF)/len(accuracy_values_RF))
print("Mean F1 Score:", sum(f1_scores_RF)/len(f1_scores_RF))
print("Mean TSS:", sum(tss_scores_RF)/len(tss_scores_RF))
print("Mean HSS:", sum(hss_scores_RF)/len(hss_scores_RF))
print("All Accuracy Values:", accuracy_values_RF)

# Before modifying Parameters
# Random Forest Results:
# Mean Accuracy: 0.621904761904762
# Mean F1 Score: 0.6178347578347578
# Mean TSS: 0.23571428571428577
# Mean HSS: 0.23738650729801175
# All Accuracy Values: [0.6666666666666666, 0.7333333333333333, 0.6, 0.4666666666666667, 0.6428571428571429]

#AFTER
# Random Forest Results:
# Mean Accuracy: 0.7171428571428572
# Mean F1 Score: 0.7145098039215687
# Mean TSS: 0.43571428571428567
# Mean HSS: 0.4344300822561692
# All Accuracy Values: [0.6666666666666666, 0.6666666666666666, 0.8666666666666667, 0.6, 0.7857142857142857]

C:\Users\shiva\AppData\Local\Temp\ipykernel_9052\1845584558.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')



Random Forest Results:
Mean Accuracy: 0.7171428571428572
Mean F1 Score: 0.7145098039215687
Mean TSS: 0.43571428571428567
Mean HSS: 0.4344300822561692
All Accuracy Values: [0.6666666666666666, 0.6666666666666666, 0.8666666666666667, 0.6, 0.7857142857142857]


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
from itertools import product

# === Load and prepare data ===
sep_data = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")

# Label data
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

# Combine data
data = pd.concat([sep_data, nsep_data])
X = data[['Sunspots', 'AR', 'Flare Class']]
y = data['label']

# Extract flare class (A/B/C/M/X)
X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')

# Convert flare class to dummy variables
flare_dummies = pd.get_dummies(X['Flare Class'], prefix='Flare')
X = pd.concat([X[['Sunspots', 'AR']], flare_dummies], axis=1)

# === Define parameter grid ===
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [200, 500],
    'class_weight': [None, 'balanced']
}

# === Generate all combinations ===
param_combinations = list(product(
    param_grid_lr['C'],
    param_grid_lr['penalty'],
    param_grid_lr['solver'],
    param_grid_lr['max_iter'],
    param_grid_lr['class_weight']
))

results = []

# === 5-fold cross-validation setup ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# === Grid search loop ===
for i, (C, penalty, solver, max_iter, class_weight) in enumerate(param_combinations):
    acc_scores, tss_scores, hss_scores = [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        try:
            model = LogisticRegression(
                C=C,
                penalty=penalty,
                solver=solver,
                max_iter=max_iter,
                class_weight=class_weight
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # Accuracy
            acc = accuracy_score(y_test, y_pred)
            acc_scores.append(acc)

            # Confusion matrix
            tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=['NSEP', 'SEP']).ravel()

            # TSS
            tss = (tp / (tp + fn)) - (fp / (fp + tn))
            tss_scores.append(tss)

            # HSS
            hss = (2 * ((tp * tn) - (fn * fp))) / (
                ((tp + fn) * (fn + tn)) + ((fp + tn) * (tp + fp))
            )
            hss_scores.append(hss)
        except Exception as e:
            print(f"⚠️ Skipped combination {i} due to error: {e}")
            continue

    results.append({
        'C': C,
        'penalty': penalty,
        'solver': solver,
        'max_iter': max_iter,
        'class_weight': class_weight,
        'Mean Accuracy': np.mean(acc_scores),
        'Mean TSS': np.mean(tss_scores),
        'Mean HSS': np.mean(hss_scores)
    })

# === Results summary ===
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Mean Accuracy', ascending=False)

# Display top 10
print("\nTop 10 Logistic Regression Configurations:\n")
print(results_df.head(10).to_string(index=False))


C:\Users\shiva\AppData\Local\Temp\ipykernel_9052\561019613.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')



Top 10 Logistic Regression Configurations:

  C penalty    solver  max_iter class_weight  Mean Accuracy  Mean TSS  Mean HSS
1.0      l2 liblinear       200     balanced       0.716190  0.428571  0.430464
1.0      l2 liblinear       500     balanced       0.716190  0.428571  0.430464
1.0      l2     lbfgs       200         None       0.716190  0.428571  0.430464
1.0      l2     lbfgs       200     balanced       0.716190  0.428571  0.430464
1.0      l2     lbfgs       500         None       0.716190  0.428571  0.430464
1.0      l2     lbfgs       500     balanced       0.716190  0.428571  0.430464
1.0      l2 liblinear       200         None       0.702857  0.403571  0.404402
1.0      l2 liblinear       500         None       0.702857  0.403571  0.404402
0.1      l2 liblinear       200     balanced       0.688571  0.375000  0.374467
0.1      l2 liblinear       500     balanced       0.688571  0.375000  0.374467


In [5]:
# for Ensemble

import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

# Concatenate dataframes
data = pd.concat([sep_data, nsep_data])

# Extract features and target
X = data[['Sunspots', 'AR', 'Flare Class']]
y = data['label']

# Extract Flare Class types without intensities
X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')

# Create dummy variables for Flare Class
flare_dummies = pd.get_dummies(X['Flare Class'], prefix='Flare')
X = pd.concat([X[['Sunspots', 'AR']], flare_dummies], axis=1)


C:\Users\shiva\AppData\Local\Temp\ipykernel_9052\999353363.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Flare Class'] = X['Flare Class'].str.extract(r'([A-Z])')


# New Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report
)

sep_data  = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")

sep_data['label']  = 'SEP'
nsep_data['label'] = 'NSEP'

train_data = pd.concat([sep_data, nsep_data], ignore_index=True)

# 'Sunspots' is absent in the new dataset.
# To keep Sunspots, change USE_SUNSPOTS = True and
# the new data will have Sunspots filled with the training mean.
USE_SUNSPOTS = False

if USE_SUNSPOTS:
    feature_cols = ['Sunspots', 'AR', 'Flare Class']
else:
    feature_cols = ['AR', 'Flare Class']

X_train_raw = train_data[feature_cols].copy()
y_train     = train_data['label']

# Extract flare class letter (A/B/C/M/X)
X_train_raw['Flare Class'] = X_train_raw['Flare Class'].str.extract(r'([A-Z])')

flare_dummies_train = pd.get_dummies(X_train_raw['Flare Class'], prefix='Flare')
X_train = pd.concat([X_train_raw.drop(columns=['Flare Class']), flare_dummies_train], axis=1)

print("Training features:", list(X_train.columns))
print(f"Training samples  : {len(X_train)}  (SEP={sum(y_train=='SEP')}, NSEP={sum(y_train=='NSEP')})\n")

rf_classifier = RandomForestClassifier(
    n_estimators=50, max_depth=6, min_samples_split=4,
    min_samples_leaf=3, max_features="sqrt",
    class_weight="balanced", random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_list, f1_list, tss_list, hss_list = [], [], [], []

for train_idx, test_idx in skf.split(X_train, y_train):
    Xtr, Xte = X_train.iloc[train_idx], X_train.iloc[test_idx]
    ytr, yte = y_train.iloc[train_idx], y_train.iloc[test_idx]

    rf_classifier.fit(Xtr, ytr)
    ypred = rf_classifier.predict(Xte)

    acc_list.append(accuracy_score(yte, ypred))
    f1_list.append(f1_score(yte, ypred, average='weighted'))

    tn, fp, fn, tp = confusion_matrix(yte, ypred, labels=['NSEP', 'SEP']).ravel()
    tss_list.append((tp/(tp+fn)) - (fp/(fp+tn)))
    hss_list.append(
        (2*((tp*tn)-(fn*fp))) / (((tp+fn)*(fn+tn)) + ((fp+tn)*(tp+fp)))
    )

print("=== 5-Fold CV on Original Data ===")
print(f"Mean Accuracy : {np.mean(acc_list):.4f}")
print(f"Mean F1       : {np.mean(f1_list):.4f}")
print(f"Mean TSS      : {np.mean(tss_list):.4f}")
print(f"Mean HSS      : {np.mean(hss_list):.4f}\n")

rf_classifier.fit(X_train, y_train)
print("Final model trained on all original data.\n")

new_data = pd.read_csv("data/New_Tabular_Data.csv")

# Map columns to match original feature names
#   xrsclass         → Flare Class  (e.g. "M1.1")
#   noaa_active_region → AR
#   SEP_Server_Association == "ProtonFlare" → SEP, else NSEP (ground truth for evaluation)
new_data['Flare Class'] = new_data['xrsclass']
new_data['AR']          = new_data['noaa_active_region']
new_data['true_label']  = new_data['SEP_Server_Association'].apply(
    lambda x: 'SEP' if x == 'ProtonFlare' else 'NSEP'
)

X_new_raw = new_data[['AR', 'Flare Class']].copy()

if USE_SUNSPOTS:
    sunspot_mean = X_train_raw['Sunspots'].mean()
    X_new_raw.insert(0, 'Sunspots', sunspot_mean)  # fill with training mean
    print(f"[INFO] Sunspots filled with training mean: {sunspot_mean:.1f}")

X_new_raw['Flare Class'] = X_new_raw['Flare Class'].str.extract(r'([A-Z])')

flare_dummies_new = pd.get_dummies(X_new_raw['Flare Class'], prefix='Flare')
X_new = pd.concat([X_new_raw.drop(columns=['Flare Class']), flare_dummies_new], axis=1)

# Align columns with training set (add missing, drop extra)
X_new = X_new.reindex(columns=X_train.columns, fill_value=0)

print(f"New dataset samples: {len(X_new)}  "
      f"(SEP={sum(new_data['true_label']=='SEP')}, "
      f"NSEP={sum(new_data['true_label']=='NSEP')})\n")


y_new_true = new_data['true_label']
y_new_pred = rf_classifier.predict(X_new)

acc  = accuracy_score(y_new_true, y_new_pred)
f1   = f1_score(y_new_true, y_new_pred, average='weighted')
tn, fp, fn, tp = confusion_matrix(y_new_true, y_new_pred, labels=['NSEP', 'SEP']).ravel()
tss  = (tp/(tp+fn)) - (fp/(fp+tn))
hss  = (2*((tp*tn)-(fn*fp))) / (((tp+fn)*(fn+tn)) + ((fp+tn)*(tp+fp)))

print("=== Results on New Dataset ===")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"TSS      : {tss:.4f}")
print(f"HSS      : {hss:.4f}")
print(f"\nConfusion Matrix (rows=true, cols=pred):")
print(f"              Pred NSEP  Pred SEP")
print(f"True NSEP :   {tn:>9}  {fp:>8}")
print(f"True SEP  :   {fn:>9}  {tp:>8}")
print("\nFull Classification Report:")
print(classification_report(y_new_true, y_new_pred, target_names=['NSEP', 'SEP']))

Training features: ['AR', 'Flare_C', 'Flare_M', 'Flare_X']
Training samples  : 74  (SEP=37, NSEP=37)

=== 5-Fold CV on Original Data ===
Mean Accuracy : 0.6076
Mean F1       : 0.6001
Mean TSS      : 0.2071
Mean HSS      : 0.2083

Final model trained on all original data.

New dataset samples: 17794  (SEP=138, NSEP=17656)

=== Results on New Dataset ===
Accuracy : 0.0419
F1 Score : 0.0679
TSS      : -0.0950
HSS      : -0.0015

Confusion Matrix (rows=true, cols=pred):
              Pred NSEP  Pred SEP
True NSEP :         625     17031
True SEP  :          18       120

Full Classification Report:
              precision    recall  f1-score   support

        NSEP       0.97      0.04      0.07     17656
         SEP       0.01      0.87      0.01       138

    accuracy                           0.04     17794
   macro avg       0.49      0.45      0.04     17794
weighted avg       0.96      0.04      0.07     17794



In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report
)

sep_data  = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")

sep_data['label']  = 'SEP'
nsep_data['label'] = 'NSEP'

train_data = pd.concat([sep_data, nsep_data], ignore_index=True)

# 'Sunspots' is absent in the new dataset.
# To keep Sunspots, change USE_SUNSPOTS = True and
# the new data will have Sunspots filled with the training mean.
USE_SUNSPOTS = True

if USE_SUNSPOTS:
    feature_cols = ['Sunspots', 'AR', 'Flare Class']
else:
    feature_cols = ['AR', 'Flare Class']

X_train_raw = train_data[feature_cols].copy()
y_train     = train_data['label']

# Extract flare class letter (A/B/C/M/X)
X_train_raw['Flare Class'] = X_train_raw['Flare Class'].str.extract(r'([A-Z])')

flare_dummies_train = pd.get_dummies(X_train_raw['Flare Class'], prefix='Flare')
X_train = pd.concat([X_train_raw.drop(columns=['Flare Class']), flare_dummies_train], axis=1)

print("Training features:", list(X_train.columns))
print(f"Training samples  : {len(X_train)}  (SEP={sum(y_train=='SEP')}, NSEP={sum(y_train=='NSEP')})\n")

rf_classifier = RandomForestClassifier(
    n_estimators=50, max_depth=6, min_samples_split=4,
    min_samples_leaf=3, max_features="sqrt",
    class_weight="balanced", random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_list, f1_list, tss_list, hss_list = [], [], [], []

for train_idx, test_idx in skf.split(X_train, y_train):
    Xtr, Xte = X_train.iloc[train_idx], X_train.iloc[test_idx]
    ytr, yte = y_train.iloc[train_idx], y_train.iloc[test_idx]

    rf_classifier.fit(Xtr, ytr)
    ypred = rf_classifier.predict(Xte)

    acc_list.append(accuracy_score(yte, ypred))
    f1_list.append(f1_score(yte, ypred, average='weighted'))

    tn, fp, fn, tp = confusion_matrix(yte, ypred, labels=['NSEP', 'SEP']).ravel()
    tss_list.append((tp/(tp+fn)) - (fp/(fp+tn)))
    hss_list.append(
        (2*((tp*tn)-(fn*fp))) / (((tp+fn)*(fn+tn)) + ((fp+tn)*(tp+fp)))
    )

print("=== 5-Fold CV on Original Data ===")
print(f"Mean Accuracy : {np.mean(acc_list):.4f}")
print(f"Mean F1       : {np.mean(f1_list):.4f}")
print(f"Mean TSS      : {np.mean(tss_list):.4f}")
print(f"Mean HSS      : {np.mean(hss_list):.4f}\n")

rf_classifier.fit(X_train, y_train)
print("Final model trained on all original data.\n")

new_data = pd.read_csv("data/New_Tabular_Data.csv")

# Map columns to match original feature names
#   xrsclass         → Flare Class  (e.g. "M1.1")
#   noaa_active_region → AR
#   SEP_Server_Association == "ProtonFlare" → SEP, else NSEP (ground truth for evaluation)
new_data['Flare Class'] = new_data['xrsclass']
new_data['AR']          = new_data['noaa_active_region']
new_data['true_label']  = new_data['SEP_Server_Association'].apply(
    lambda x: 'SEP' if x == 'ProtonFlare' else 'NSEP'
)

X_new_raw = new_data[['AR', 'Flare Class']].copy()

if USE_SUNSPOTS:
    sunspot_mean = X_train_raw['Sunspots'].mean()
    X_new_raw.insert(0, 'Sunspots', sunspot_mean)  # fill with training mean
    print(f"[INFO] Sunspots filled with training mean: {sunspot_mean:.1f}")

X_new_raw['Flare Class'] = X_new_raw['Flare Class'].str.extract(r'([A-Z])')

flare_dummies_new = pd.get_dummies(X_new_raw['Flare Class'], prefix='Flare')
X_new = pd.concat([X_new_raw.drop(columns=['Flare Class']), flare_dummies_new], axis=1)

# Align columns with training set (add missing, drop extra)
X_new = X_new.reindex(columns=X_train.columns, fill_value=0)

print(f"New dataset samples: {len(X_new)}  "
      f"(SEP={sum(new_data['true_label']=='SEP')}, "
      f"NSEP={sum(new_data['true_label']=='NSEP')})\n")


y_new_true = new_data['true_label']
y_new_pred = rf_classifier.predict(X_new)

acc  = accuracy_score(y_new_true, y_new_pred)
f1   = f1_score(y_new_true, y_new_pred, average='weighted')
tn, fp, fn, tp = confusion_matrix(y_new_true, y_new_pred, labels=['NSEP', 'SEP']).ravel()
tss  = (tp/(tp+fn)) - (fp/(fp+tn))
hss  = (2*((tp*tn)-(fn*fp))) / (((tp+fn)*(fn+tn)) + ((fp+tn)*(tp+fp)))

print("=== Results on New Dataset ===")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"TSS      : {tss:.4f}")
print(f"HSS      : {hss:.4f}")
print(f"\nConfusion Matrix (rows=true, cols=pred):")
print(f"              Pred NSEP  Pred SEP")
print(f"True NSEP :   {tn:>9}  {fp:>8}")
print(f"True SEP  :   {fn:>9}  {tp:>8}")
print("\nFull Classification Report:")
print(classification_report(y_new_true, y_new_pred, target_names=['NSEP', 'SEP']))

Training features: ['Sunspots', 'AR', 'Flare_C', 'Flare_M', 'Flare_X']
Training samples  : 74  (SEP=37, NSEP=37)

=== 5-Fold CV on Original Data ===
Mean Accuracy : 0.7171
Mean F1       : 0.7145
Mean TSS      : 0.4357
Mean HSS      : 0.4344

Final model trained on all original data.

[INFO] Sunspots filled with training mean: 24.7
New dataset samples: 17794  (SEP=138, NSEP=17656)

=== Results on New Dataset ===
Accuracy : 0.4916
F1 Score : 0.6514
TSS      : 0.2863
HSS      : 0.0086

Confusion Matrix (rows=true, cols=pred):
              Pred NSEP  Pred SEP
True NSEP :        8637      9019
True SEP  :          28       110

Full Classification Report:
              precision    recall  f1-score   support

        NSEP       1.00      0.49      0.66     17656
         SEP       0.01      0.80      0.02       138

    accuracy                           0.49     17794
   macro avg       0.50      0.64      0.34     17794
weighted avg       0.99      0.49      0.65     17794



In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

# APPROACH A: Train on original data, test on balanced new data
print("Method #2: Original model → Balanced new data")

# Load & prep original training data
sep_data  = pd.read_csv("data/SEP_Tabular_6h.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h.csv")
sep_data['label']  = 'SEP'
nsep_data['label'] = 'NSEP'
train_data = pd.concat([sep_data, nsep_data], ignore_index=True)

def prepare_features(df, feature_col='Flare Class', ar_col='AR', fit_columns=None):
    X_raw = df[[ar_col, feature_col]].copy()
    X_raw[feature_col] = X_raw[feature_col].str.extract(r'([A-Z])')
    dummies = pd.get_dummies(X_raw[feature_col], prefix='Flare')
    X = pd.concat([X_raw[[ar_col]].rename(columns={ar_col: 'AR'}), dummies], axis=1)
    if fit_columns is not None:
        X = X.reindex(columns=fit_columns, fill_value=0)
    return X

X_train = prepare_features(train_data)
y_train = train_data['label']
train_columns = X_train.columns

rf_orig = RandomForestClassifier(
    n_estimators=50, max_depth=6, min_samples_split=4,
    min_samples_leaf=3, max_features="sqrt",
    class_weight="balanced", random_state=42
)
rf_orig.fit(X_train, y_train)

# Load & balance new data
new_data = pd.read_csv("data/New_Tabular_Data.csv")
new_data['true_label'] = new_data['SEP_Server_Association'].apply(
    lambda x: 'SEP' if x == 'ProtonFlare' else 'NSEP'
)
sep_new  = new_data[new_data['true_label'] == 'SEP']
nsep_new = new_data[new_data['true_label'] == 'NSEP']
nsep_sampled = nsep_new.sample(n=138, random_state=42)
balanced_new = pd.concat([sep_new, nsep_sampled], ignore_index=True)

print(f"Balanced new dataset: {len(balanced_new)} samples "
      f"(SEP={len(sep_new)}, NSEP={len(nsep_sampled)})\n")

# Map new data columns to match original feature names
balanced_new = balanced_new.rename(columns={
    'xrsclass': 'Flare Class',
    'noaa_active_region': 'AR'
})

X_new = prepare_features(balanced_new, fit_columns=train_columns)
y_new = balanced_new['true_label']
y_pred_a = rf_orig.predict(X_new)

acc  = accuracy_score(y_new, y_pred_a)
f1   = f1_score(y_new, y_pred_a, average='weighted')
tn, fp, fn, tp = confusion_matrix(y_new, y_pred_a, labels=['NSEP', 'SEP']).ravel()
tss  = (tp/(tp+fn)) - (fp/(fp+tn))
hss  = (2*((tp*tn)-(fn*fp))) / (((tp+fn)*(fn+tn)) + ((fp+tn)*(tp+fp)))

print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"TSS      : {tss:.4f}")
print(f"HSS      : {hss:.4f}")
print(f"\nConfusion Matrix:")
print(f"              Pred NSEP  Pred SEP")
print(f"True NSEP :   {tn:>9}  {fp:>8}")
print(f"True SEP  :   {fn:>9}  {tp:>8}")
print("\n" + classification_report(y_new, y_pred_a, target_names=['NSEP', 'SEP']))

Method #2: Original model → Balanced new data
Balanced new dataset: 276 samples (SEP=138, NSEP=138)

Accuracy : 0.4529
F1 Score : 0.3380
TSS      : -0.0942
HSS      : -0.0942

Confusion Matrix:
              Pred NSEP  Pred SEP
True NSEP :           5       133
True SEP  :          18       120

              precision    recall  f1-score   support

        NSEP       0.22      0.04      0.06       138
         SEP       0.47      0.87      0.61       138

    accuracy                           0.45       276
   macro avg       0.35      0.45      0.34       276
weighted avg       0.35      0.45      0.34       276



In [ ]:
print("Method #3: Retrain on balanced new data (5-Fold CV)")

balanced_new_shuffled = balanced_new.sample(frac=1, random_state=42).reset_index(drop=True)
X_b = prepare_features(balanced_new_shuffled)
y_b = balanced_new_shuffled['true_label']

print(f"Training features: {list(X_b.columns)}")
print(f"Flare class breakdown: "
      f"{balanced_new_shuffled['Flare Class'].str.extract(r'([A-Z])')[0].value_counts().to_dict()}\n")

rf_new = RandomForestClassifier(
    n_estimators=50, max_depth=6, min_samples_split=4,
    min_samples_leaf=3, max_features="sqrt",
    class_weight="balanced", random_state=42
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_l, f1_l, tss_l, hss_l = [], [], [], []
fold_results = []

for i, (train_idx, test_idx) in enumerate(skf.split(X_b, y_b), 1):
    Xtr, Xte = X_b.iloc[train_idx], X_b.iloc[test_idx]
    ytr, yte = y_b.iloc[train_idx], y_b.iloc[test_idx]
    rf_new.fit(Xtr, ytr)
    yp = rf_new.predict(Xte)

    a   = accuracy_score(yte, yp)
    f   = f1_score(yte, yp, average='weighted')
    tn2, fp2, fn2, tp2 = confusion_matrix(yte, yp, labels=['NSEP', 'SEP']).ravel()
    t   = (tp2/(tp2+fn2)) - (fp2/(fp2+tn2))
    h   = (2*((tp2*tn2)-(fn2*fp2))) / (((tp2+fn2)*(fn2+tn2)) + ((fp2+tn2)*(tp2+fp2)))

    acc_l.append(a); f1_l.append(f); tss_l.append(t); hss_l.append(h)
    fold_results.append({'Fold': i, 'Accuracy': round(a,4), 'F1': round(f,4),
                         'TSS': round(t,4), 'HSS': round(h,4)})

print(pd.DataFrame(fold_results).to_string(index=False))
print(f"\nMean Accuracy : {np.mean(acc_l):.4f}")
print(f"Mean F1       : {np.mean(f1_l):.4f}")
print(f"Mean TSS      : {np.mean(tss_l):.4f}")
print(f"Mean HSS      : {np.mean(hss_l):.4f}")

Method #3: Retrain on balanced new data (5-Fold CV)
Training features: ['AR', 'Flare_C', 'Flare_M', 'Flare_X']
Flare class breakdown: {'C': 154, 'M': 78, 'X': 44}

 Fold  Accuracy     F1    TSS    HSS
    1    0.8750 0.8730 0.7500 0.7500
    2    0.8545 0.8543 0.7077 0.7086
    3    0.8545 0.8544 0.7103 0.7094
    4    0.8909 0.8902 0.7791 0.7812
    5    0.9091 0.9087 0.8161 0.8178

Mean Accuracy : 0.8768
Mean F1       : 0.8761
Mean TSS      : 0.7526
Mean HSS      : 0.7534


# New Features

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h_enriched.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h_enriched.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

# Concatenate dataframes
data = pd.concat([sep_data, nsep_data], ignore_index=True)

# --- Feature Engineering ---

# Extract flare class letters and one-hot encode
data['Flare Class Letter'] = data['Flare Class'].str.extract(r'([A-Z])')
flare_dummies = pd.get_dummies(data['Flare Class Letter'], prefix='Flare')

# Base features
base_features = data[['Sunspots', 'AR']].copy()

# Solar wind features (new)
sw_features = data[['SW Temp', 'SW Velocity', 'SW Density']].copy()

# Combine all features
X = pd.concat([base_features, flare_dummies, sw_features], axis=1)

# Ensure all columns are numeric
X = X.apply(pd.to_numeric, errors='coerce')

y = data['label']

# Report how many rows have missing SW values
sw_missing = X[['SW Temp', 'SW Velocity', 'SW Density']].isna().any(axis=1).sum()
print(f"Rows with missing SW data: {sw_missing}/{len(X)} (will be imputed with column mean)")

# Fill missing SW values with column mean (events that predate MEMPSEP coverage)
X = X.fillna(X.mean(numeric_only=True))

# --- Classifier ---
rf_classifier = RandomForestClassifier(
    n_estimators=50,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

# --- Cross-validation ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_values_RF = []
f1_scores_RF = []
tss_scores_RF = []
hss_scores_RF = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_classifier.fit(X_train, y_train)
    y_pred = rf_classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=['SEP', 'NSEP']).ravel()
    tss = (tp / (tp + fn)) - (fp / (fp + tn))
    hss = (2 * ((tp * tn) - (fn * fp))) / (((tp + fn) * (fn + tn)) + ((fp + tn) * (tp + fp)))

    accuracy_values_RF.append(accuracy)
    f1_scores_RF.append(f1)
    tss_scores_RF.append(tss)
    hss_scores_RF.append(hss)

# --- Feature Importance ---
rf_classifier.fit(X, y)  # refit on full data for importance ranking
feature_importance = pd.Series(rf_classifier.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False)

# --- Results ---
print("\nRandom Forest Results:")
print(f"Mean Accuracy : {sum(accuracy_values_RF)/len(accuracy_values_RF):.4f}")
print(f"Mean F1 Score : {sum(f1_scores_RF)/len(f1_scores_RF):.4f}")
print(f"Mean TSS      : {sum(tss_scores_RF)/len(tss_scores_RF):.4f}")
print(f"Mean HSS      : {sum(hss_scores_RF)/len(hss_scores_RF):.4f}")
print(f"All Accuracy Values: {[round(v, 4) for v in accuracy_values_RF]}")

print("\nFeature Importances:")
for feat, imp in feature_importance.items():
    print(f"  {feat:<20} {imp:.4f}")

Rows with missing SW data: 17/74 (will be imputed with column mean)

Random Forest Results:
Mean Accuracy : 0.6486
Mean F1 Score : 0.6430
Mean TSS      : 0.2929
Mean HSS      : 0.2952
All Accuracy Values: [0.5333, 0.7333, 0.8, 0.5333, 0.6429]

Feature Importances:
  Sunspots             0.1853
  Flare_M              0.1674
  AR                   0.1417
  SW Temp              0.1402
  SW Velocity          0.1263
  SW Density           0.1253
  Flare_X              0.1029
  Flare_C              0.0109


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h_enriched.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h_enriched.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

# Concatenate dataframes
data = pd.concat([sep_data, nsep_data], ignore_index=True)

data['Flare Class Letter'] = data['Flare Class'].str.extract(r'([A-Z])')
flare_dummies = pd.get_dummies(data['Flare Class Letter'], prefix='Flare')

base_features   = data[['Sunspots', 'AR']].copy()
sw_features     = data[['SW Temp', 'SW Velocity', 'SW Density']].copy()
imf_features    = data[['IMF Bx', 'IMF By', 'IMF Bz']].copy()

feature_sets = {
    "SS + AR + FC":               pd.concat([base_features, flare_dummies], axis=1),
    "SS + AR + FC + SW":          pd.concat([base_features, flare_dummies, sw_features], axis=1),
    "SS + AR + FC + IMF Bx/By/Bz": pd.concat([base_features, flare_dummies, imf_features], axis=1),
}

y = data['label']

rf_classifier = RandomForestClassifier(
    n_estimators=50,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for set_name, X_raw in feature_sets.items():

    X = X_raw.apply(pd.to_numeric, errors='coerce')
    missing = X.isna().any(axis=1).sum()
    if missing > 0:
        print(f"[{set_name}] {missing} rows with missing values — imputed with column mean.")
    X = X.fillna(X.mean(numeric_only=True))

    accuracy_values_RF = []
    f1_scores_RF       = []
    tss_scores_RF      = []
    hss_scores_RF      = []

    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        rf_classifier.fit(X_train, y_train)
        y_pred = rf_classifier.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=['SEP', 'NSEP']).ravel()
        tss = (tp / (tp + fn)) - (fp / (fp + tn))
        hss = (2 * ((tp * tn) - (fn * fp))) / (((tp + fn) * (fn + tn)) + ((fp + tn) * (tp + fp)))

        accuracy_values_RF.append(accuracy)
        f1_scores_RF.append(f1)
        tss_scores_RF.append(tss)
        hss_scores_RF.append(hss)

    rf_classifier.fit(X, y)
    feature_importance = pd.Series(rf_classifier.feature_importances_, index=X.columns)
    feature_importance = feature_importance.sort_values(ascending=False)

    print(f"\n{'='*50}")
    print(f"Random Forest Results — {set_name}")
    print(f"{'='*50}")
    print(f"Mean Accuracy : {sum(accuracy_values_RF)/len(accuracy_values_RF):.4f}")
    print(f"Mean F1 Score : {sum(f1_scores_RF)/len(f1_scores_RF):.4f}")
    print(f"Mean TSS      : {sum(tss_scores_RF)/len(tss_scores_RF):.4f}")
    print(f"Mean HSS      : {sum(hss_scores_RF)/len(hss_scores_RF):.4f}")
    print(f"All Accuracy Values: {[round(v, 4) for v in accuracy_values_RF]}")
    print(f"\nFeature Importances:")
    for feat, imp in feature_importance.items():
        print(f"  {feat:<25} {imp:.4f}")


Random Forest Results — SS + AR + FC
Mean Accuracy : 0.7171
Mean F1 Score : 0.7145
Mean TSS      : 0.4357
Mean HSS      : 0.4344
All Accuracy Values: [0.6667, 0.6667, 0.8667, 0.6, 0.7857]

Feature Importances:
  Sunspots                  0.3392
  AR                        0.3117
  Flare_X                   0.1781
  Flare_M                   0.1586
  Flare_C                   0.0123
[SS + AR + FC + SW] 17 rows with missing values — imputed with column mean.

Random Forest Results — SS + AR + FC + SW
Mean Accuracy : 0.6486
Mean F1 Score : 0.6430
Mean TSS      : 0.2929
Mean HSS      : 0.2952
All Accuracy Values: [0.5333, 0.7333, 0.8, 0.5333, 0.6429]

Feature Importances:
  Sunspots                  0.1853
  Flare_M                   0.1674
  AR                        0.1417
  SW Temp                   0.1402
  SW Velocity               0.1263
  SW Density                0.1253
  Flare_X                   0.1029
  Flare_C                   0.0109
[SS + AR + FC + IMF Bx/By/Bz] 7 rows with 

In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h_enriched.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h_enriched.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

data = pd.concat([sep_data, nsep_data], ignore_index=True)

data['Flare Class Letter'] = data['Flare Class'].str.extract(r'([A-Z])')
flare_dummies = pd.get_dummies(data['Flare Class Letter'], prefix='Flare')

base_features    = data[['Sunspots', 'AR']].copy()
sw_features      = data[['SW Temp', 'SW Velocity', 'SW Density']].copy()
imf_features     = data[['IMF Bx', 'IMF By', 'IMF Bz']].copy()
fluence_feature  = data[['Fluence']].copy()
duration_feature = data[['Duration']].copy()

feature_sets = {
    "SS + AR + FC":                      pd.concat([base_features, flare_dummies], axis=1),
    "SS + AR + FC + SW":                 pd.concat([base_features, flare_dummies, sw_features], axis=1),
    "SS + AR + FC + IMF Bx/By/Bz":      pd.concat([base_features, flare_dummies, imf_features], axis=1),
    "SS + AR + FC + Fluence":            pd.concat([base_features, flare_dummies, fluence_feature], axis=1),
    "SS + AR + FC + Fluence + Duration": pd.concat([base_features, flare_dummies, fluence_feature, duration_feature], axis=1),
}

y = data['label']

rf_classifier = RandomForestClassifier(
    n_estimators=50,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for set_name, X_raw in feature_sets.items():

    X = X_raw.apply(pd.to_numeric, errors='coerce')
    missing = X.isna().any(axis=1).sum()
    if missing > 0:
        print(f"[{set_name}] {missing} rows with missing values — imputed with column mean.")
    X = X.fillna(X.mean(numeric_only=True))

    accuracy_values_RF = []
    f1_scores_RF       = []
    tss_scores_RF      = []
    hss_scores_RF      = []

    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        rf_classifier.fit(X_train, y_train)
        y_pred = rf_classifier.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=['SEP', 'NSEP']).ravel()
        tss = (tp / (tp + fn)) - (fp / (fp + tn))
        hss = (2 * ((tp * tn) - (fn * fp))) / (((tp + fn) * (fn + tn)) + ((fp + tn) * (tp + fp)))

        accuracy_values_RF.append(accuracy)
        f1_scores_RF.append(f1)
        tss_scores_RF.append(tss)
        hss_scores_RF.append(hss)

    rf_classifier.fit(X, y)
    feature_importance = pd.Series(rf_classifier.feature_importances_, index=X.columns)
    feature_importance = feature_importance.sort_values(ascending=False)

    print(f"\n{'='*50}")
    print(f"Random Forest Results — {set_name}")
    print(f"{'='*50}")
    print(f"Mean Accuracy : {sum(accuracy_values_RF)/len(accuracy_values_RF):.4f}")
    print(f"Mean F1 Score : {sum(f1_scores_RF)/len(f1_scores_RF):.4f}")
    print(f"Mean TSS      : {sum(tss_scores_RF)/len(tss_scores_RF):.4f}")
    print(f"Mean HSS      : {sum(hss_scores_RF)/len(hss_scores_RF):.4f}")
    print(f"All Accuracy Values: {[round(v, 4) for v in accuracy_values_RF]}")
    print(f"\nFeature Importances:")
    for feat, imp in feature_importance.items():
        print(f"  {feat:<25} {imp:.4f}")


Random Forest Results — SS + AR + FC
Mean Accuracy : 0.7171
Mean F1 Score : 0.7145
Mean TSS      : 0.4357
Mean HSS      : 0.4344
All Accuracy Values: [0.6667, 0.6667, 0.8667, 0.6, 0.7857]

Feature Importances:
  Sunspots                  0.3392
  AR                        0.3117
  Flare_X                   0.1781
  Flare_M                   0.1586
  Flare_C                   0.0123
[SS + AR + FC + SW] 17 rows with missing values — imputed with column mean.

Random Forest Results — SS + AR + FC + SW
Mean Accuracy : 0.6486
Mean F1 Score : 0.6430
Mean TSS      : 0.2929
Mean HSS      : 0.2952
All Accuracy Values: [0.5333, 0.7333, 0.8, 0.5333, 0.6429]

Feature Importances:
  Sunspots                  0.1853
  Flare_M                   0.1674
  AR                        0.1417
  SW Temp                   0.1402
  SW Velocity               0.1263
  SW Density                0.1253
  Flare_X                   0.1029
  Flare_C                   0.0109
[SS + AR + FC + IMF Bx/By/Bz] 7 rows with 

In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Load data
sep_data = pd.read_csv("data/SEP_Tabular_6h_enriched.csv")
nsep_data = pd.read_csv("data/NSEP_Tabular_6h_enriched.csv")

# Add target labels
sep_data['label'] = 'SEP'
nsep_data['label'] = 'NSEP'

data = pd.concat([sep_data, nsep_data], ignore_index=True)


data['Flare Class Letter'] = data['Flare Class'].str.extract(r'([A-Z])')
flare_dummies = pd.get_dummies(data['Flare Class Letter'], prefix='Flare')

base_features    = data[['Sunspots', 'AR']].copy()
sw_features      = data[['SW Temp', 'SW Velocity', 'SW Density']].copy()
imf_features     = data[['IMF Bx', 'IMF By', 'IMF Bz']].copy()
fluence_feature  = data[['Fluence']].copy()
duration_feature = data[['Duration']].copy()

feature_sets = {
    "SS + AR + FC":                      pd.concat([base_features, flare_dummies], axis=1),

    "SS + AR + FC + SW Temp":            pd.concat([base_features, flare_dummies, data[['SW Temp']]], axis=1),
    "SS + AR + FC + SW Velocity":        pd.concat([base_features, flare_dummies, data[['SW Velocity']]], axis=1),
    "SS + AR + FC + SW Density":         pd.concat([base_features, flare_dummies, data[['SW Density']]], axis=1),

    "SS + AR + FC + IMF Bx":             pd.concat([base_features, flare_dummies, data[['IMF Bx']]], axis=1),
    "SS + AR + FC + IMF By":             pd.concat([base_features, flare_dummies, data[['IMF By']]], axis=1),
    "SS + AR + FC + IMF Bz":             pd.concat([base_features, flare_dummies, data[['IMF Bz']]], axis=1),

    "SS + AR + FC + SW":                 pd.concat([base_features, flare_dummies, sw_features], axis=1),
    "SS + AR + FC + IMF Bx/By/Bz":      pd.concat([base_features, flare_dummies, imf_features], axis=1),
    "SS + AR + FC + Fluence":            pd.concat([base_features, flare_dummies, fluence_feature], axis=1),
    "SS + AR + FC + Fluence + Duration": pd.concat([base_features, flare_dummies, fluence_feature, duration_feature], axis=1),
}

y = data['label']

rf_classifier = RandomForestClassifier(
    n_estimators=50,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for set_name, X_raw in feature_sets.items():

    X = X_raw.apply(pd.to_numeric, errors='coerce')
    missing = X.isna().any(axis=1).sum()
    if missing > 0:
        print(f"[{set_name}] {missing} rows with missing values — imputed with column mean.")
    X = X.fillna(X.mean(numeric_only=True))

    accuracy_values_RF = []
    f1_scores_RF       = []
    tss_scores_RF      = []
    hss_scores_RF      = []

    for train_index, test_index in skf.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        rf_classifier.fit(X_train, y_train)
        y_pred = rf_classifier.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=['SEP', 'NSEP']).ravel()
        tss = (tp / (tp + fn)) - (fp / (fp + tn))
        hss = (2 * ((tp * tn) - (fn * fp))) / (((tp + fn) * (fn + tn)) + ((fp + tn) * (tp + fp)))

        accuracy_values_RF.append(accuracy)
        f1_scores_RF.append(f1)
        tss_scores_RF.append(tss)
        hss_scores_RF.append(hss)

    rf_classifier.fit(X, y)
    feature_importance = pd.Series(rf_classifier.feature_importances_, index=X.columns)
    feature_importance = feature_importance.sort_values(ascending=False)

    print(f"\n{'='*50}")
    print(f"Random Forest Results — {set_name}")
    print(f"{'='*50}")
    print(f"Mean Accuracy : {sum(accuracy_values_RF)/len(accuracy_values_RF):.4f}")
    print(f"Mean F1 Score : {sum(f1_scores_RF)/len(f1_scores_RF):.4f}")
    print(f"Mean TSS      : {sum(tss_scores_RF)/len(tss_scores_RF):.4f}")
    print(f"Mean HSS      : {sum(hss_scores_RF)/len(hss_scores_RF):.4f}")
    print(f"All Accuracy Values: {[round(v, 4) for v in accuracy_values_RF]}")
    print(f"\nFeature Importances:")
    for feat, imp in feature_importance.items():
        print(f"  {feat:<25} {imp:.4f}")


Random Forest Results — SS + AR + FC
Mean Accuracy : 0.7171
Mean F1 Score : 0.7145
Mean TSS      : 0.4357
Mean HSS      : 0.4344
All Accuracy Values: [0.6667, 0.6667, 0.8667, 0.6, 0.7857]

Feature Importances:
  Sunspots                  0.3392
  AR                        0.3117
  Flare_X                   0.1781
  Flare_M                   0.1586
  Flare_C                   0.0123
[SS + AR + FC + SW Temp] 14 rows with missing values — imputed with column mean.

Random Forest Results — SS + AR + FC + SW Temp
Mean Accuracy : 0.6762
Mean F1 Score : 0.6731
Mean TSS      : 0.3500
Mean HSS      : 0.3504
All Accuracy Values: [0.6, 0.7333, 0.8, 0.5333, 0.7143]

Feature Importances:
  Sunspots                  0.2588
  AR                        0.2220
  SW Temp                   0.2056
  Flare_M                   0.1899
  Flare_X                   0.1205
  Flare_C                   0.0033
[SS + AR + FC + SW Velocity] 9 rows with missing values — imputed with column mean.

Random Forest Result